# Real/bogus CNN

**Goal:** train a convolutional network that looks at the science / reference / difference stamps of each detection and
says how likely it is to be a real point source, then prove it beats the simple feature baseline.

Read `docs/realbogus_cnn.md` alongside this notebook.

In [ ]:
import json
import subprocess
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import precision_recall_curve

from asthunt.config.load_config import load_config
from asthunt.dataset.make_unit_tag import make_unit_tag
from asthunt.acquire.load_unit_metadata import load_unit_metadata
from asthunt.dataset.assemble_stamp_dataset import assemble_stamp_dataset
from asthunt.dataset.balance_classes import balance_classes
from asthunt.models.train_baseline_classifier import train_baseline_classifier
from asthunt.models.train_realbogus_cnn import train_realbogus_cnn
from asthunt.models.predict_realbogus import predict_realbogus
from asthunt.models.cross_fit_realbogus_scores import cross_fit_realbogus_scores
from asthunt.evaluate.compute_classification_metrics import compute_classification_metrics
from asthunt.evaluate.choose_threshold import choose_threshold
from asthunt.evaluate.measure_completeness import measure_completeness

project = load_config("project"); paths = project["paths"]
acquire, dataset, train_settings = load_config("acquire"), load_config("dataset"), load_config("train")


def unit_of(night):
    """The original quadrant (configs/acquire.yaml: field, ccd, quadrant) on one night."""
    return {"field": acquire["field"], "ccd": acquire["ccd"], "quadrant": acquire["quadrant"], "night": night}


nights, field = acquire["nights"], acquire["field"]
device = "mps" if torch.backends.mps.is_available() else "cpu"
rng = np.random.default_rng(project["seed"])
print("device:", device)

## 1. Data: balanced training set, realistic validation set

In [ ]:
train_stamps, train_labels, train_table = assemble_stamp_dataset([unit_of(n) for n in nights["train"]], paths["processed"], rng, dataset["bogus_per_real"])
validation_stamps, validation_labels, validation_table = assemble_stamp_dataset([unit_of(n) for n in nights["validation"]], paths["processed"], rng)
print(f"train      {len(train_labels):7d} stamps, {train_labels.mean():.1%} real")
print(f"validation {len(validation_labels):7d} stamps, {validation_labels.mean():.1%} real")

In [ ]:
def show(stamps, titles):
    fig, axes = plt.subplots(3, len(stamps), figsize=(1.9 * len(stamps), 5.7))
    for i, stamp in enumerate(stamps):
        for channel, name in enumerate(["sci", "ref", "diff"]):
            axes[channel, i].imshow(stamp[channel], origin="lower", cmap="gray", vmin=-3, vmax=10)
            axes[channel, i].set_xticks([]); axes[channel, i].set_yticks([])
            if i == 0: axes[channel, i].set_ylabel(name)
        axes[0, i].set_title(titles[i], fontsize=8)
    plt.show()

real_rows = np.flatnonzero(train_labels == 1)[:5]
bogus_rows = np.flatnonzero(train_labels == 0)[:5]
show(train_stamps[np.concatenate([real_rows, bogus_rows])], ["real"] * 5 + ["bogus"] * 5)

## 2. Train
Each epoch: shuffle → batches of 256 → augment (rotate/flip) → forward → loss → backward → Adam step.
After each epoch: score all validation stamps; keep the best epoch; stop when it stops improving.

In [ ]:
model, history = train_realbogus_cnn(train_stamps, train_labels, validation_stamps, validation_labels,
                                     train_settings, device, project["seed"])
history = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], "o-", label="train (balanced, augmented)")
axes[0].plot(history["epoch"], history["validation_loss"], "o-", label="validation (realistic)")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
axes[1].plot(history["epoch"], history["validation_ap"], "o-"); axes[1].set_xlabel("epoch"); axes[1].set_ylabel("validation average precision")
plt.show()

The two losses are not directly comparable: training has 1 real per 5 bogus and augmented stamps; validation has the
true imbalance. Watch their **trends**, and judge with validation average precision.

## 3. CNN vs the feature baseline (same validation detections)

In [ ]:
FEATURES = ["snr", "peak", "npix", "a", "b", "flux"]
train_rows_all = pd.concat([pd.read_parquet(paths["processed"] / f"detections__{make_unit_tag(unit_of(n))}.parquet") for n in nights["train"]], ignore_index=True)
balanced = train_rows_all.iloc[balance_classes(train_rows_all["label"], dataset["bogus_per_real"], np.random.default_rng(0))]
baseline = train_baseline_classifier(balanced[FEATURES].to_numpy(), balanced["label"].to_numpy())

scores = {
    "feature baseline": baseline.predict_proba(validation_table[FEATURES].to_numpy())[:, 1],
    "CNN": predict_realbogus(model, validation_stamps, device),
}
n_frames = validation_table.groupby(["night", "frame_index"]).ngroups
for name, s in scores.items():
    precision, recall, _ = precision_recall_curve(validation_labels, s)
    plt.plot(recall, precision, label=name)
    threshold = choose_threshold(validation_labels, s, 0.90)
    m = compute_classification_metrics(validation_labels, s, threshold)
    print(f"{name:17s} AP {m['average_precision']:.3f} | at 90% recall: precision {m['precision']:.3f}, "
          f"false positives per frame {m['false_positives'] / n_frames:.0f}")
plt.xlabel("recall"); plt.ylabel("precision"); plt.legend(); plt.show()

## 4. Completeness vs magnitude at 90% recall

In [ ]:
injections = pd.concat([pd.read_parquet(paths["processed"] / f"injections__{make_unit_tag(unit_of(n))}.parquet") for n in nights["validation"]], ignore_index=True)
key = lambda t: list(zip(t["night"], t["injection_id"], t["frame_index"]))

bins = np.arange(17, 22.25, 0.25)
detected = set(key(validation_table[validation_table["injection_id"] >= 0]))
injections["detected"] = [k in detected for k in key(injections)]
curve = measure_completeness(injections, "detected", "mag", bins)
plt.plot(curve["bin_centre"], curve["completeness"], "k--", label="detected (upper limit)")
for name, s in scores.items():
    threshold = choose_threshold(validation_labels, s, 0.90)
    kept_rows = validation_table[(validation_table["injection_id"] >= 0) & (s >= threshold)]
    kept = set(key(kept_rows))
    injections["kept"] = [k in kept for k in key(injections)]
    curve = measure_completeness(injections, "kept", "mag", bins)
    plt.plot(curve["bin_centre"], curve["completeness"], "o-", label=name)
plt.xlabel("injected magnitude"); plt.ylabel("completeness"); plt.legend(); plt.show()

## 5. Look at the mistakes

In [ ]:
cnn = scores["CNN"]
worst_bogus = np.argsort(-np.where(validation_labels == 0, cnn, -1))[:6]
worst_real = np.argsort(np.where(validation_labels == 1, cnn, 2))[:6]
show(validation_stamps[worst_bogus], [f"bogus, p={cnn[i]:.2f}" for i in worst_bogus])
show(validation_stamps[worst_real], [f"real, p={cnn[i]:.2f}\nSNR {validation_table['snr'].iloc[i]:.1f}" for i in worst_real])

Look at the "bogus" detections the CNN loves. Some may be genuinely real: variable stars, or asteroids that SkyBoT doesn't know
(or that are fainter than our known-asteroid cut). That's **label noise**, and it's exactly why our final candidates
get human vetting. The "real" ones it hates are usually very faint fakes, near or below the noise.

## 6. What did the first layer learn?

In [ ]:
filters = model[0].weight.detach().cpu().numpy()          # (16 filters, 3 channels, 3, 3)
fig, axes = plt.subplots(3, 16, figsize=(16, 3.3))
for f in range(16):
    for channel in range(3):
        axes[channel, f].imshow(filters[f, channel], cmap="RdBu_r"); axes[channel, f].axis("off")
plt.suptitle("16 first-layer filters (rows: sci, ref, diff). Red = positive weight, blue = negative"); plt.show()

## 7. Save the model with its provenance

In [ ]:
git_sha = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=paths["models"].parent).stdout.strip()
model_name = f"realbogus_cnn__{date.today():%Y%m%d}__{git_sha}"
paths["models"].mkdir(exist_ok=True)
torch.save(model.state_dict(), paths["models"] / f"{model_name}.pt")
with open(paths["models"] / f"{model_name}.json", "w") as f:
    json.dump({"train_nights": nights["train"], "validation_nights": nights["validation"], "settings": train_settings,
               "bogus_per_real": dataset["bogus_per_real"], "history": history.to_dict("records")}, f, indent=1)
print("saved", model_name)

## 8. Scores for linking: cross-fitted for training nights, final model for the rest
The tracklet classifier will use these scores. Training nights get **out-of-fold** scores (from CNNs that never saw that night);
validation and test nights get the final model's scores.

In [ ]:
# Training units get OUT-OF-FOLD scores; validation and test units get the final model's scores
train_units = [unit_of(n) for n in nights["train"]]
unit_scores = cross_fit_realbogus_scores(train_units, 4, validation_stamps, validation_labels, paths["processed"],
                                         dataset["bogus_per_real"], train_settings, device, project["seed"])
for night in nights["validation"] + nights["test"]:
    stamps, _, _ = assemble_stamp_dataset([unit_of(night)], paths["processed"], rng)
    unit_scores[make_unit_tag(unit_of(night))] = predict_realbogus(model, stamps, device)

for tag, s in unit_scores.items():
    np.save(paths["processed"] / f"realbogus_scores__{tag}.npy", s)
    print(f"{tag}: {len(s)} scores saved")

## Exercises
1. Remove augmentation (pass stamps straight through in the training loop). Does validation AP drop? Does training loss fall faster?
2. Train with only the difference channel (zero the other two). How much do science and reference add?
3. Compare out-of-fold training-night scores with scores from the final model on the same nights. Which are more confident, and why?